LOAD DATASET & STUDY HEAT VARIABLES

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df = pd.read_csv("../datasets/processed/climate_anomalies.csv")

print("Dataset Loaded Successfully")
print("Shape:", df.shape)

print("\nTEMPERATURE SUMMARY")
print(df["temperature_celsius"].describe())

print("\nFEELS LIKE SUMMARY")
print(df["feels_like_celsius"].describe())

print("\nTOP 20 FEELS LIKE VALUES")
print(df["feels_like_celsius"].round(1).value_counts().head(20))

print("\nMAX TEMPERATURE")
print(df["temperature_celsius"].max())

print("\nMAX FEELS LIKE")
print(df["feels_like_celsius"].max())

print("\nMIN FEELS LIKE")
print(df["feels_like_celsius"].min())

Dataset Loaded Successfully
Shape: (119398, 149)

TEMPERATURE SUMMARY
count    119398.000000
mean         20.303723
std           6.538599
min         -30.700000
25%          16.600000
50%          21.200000
75%          24.900000
max          42.800000
Name: temperature_celsius, dtype: float64

FEELS LIKE SUMMARY
count    119398.000000
mean         20.955116
std           7.468814
min         -36.500000
25%          16.600000
50%          21.200000
75%          26.200000
max          50.500000
Name: feels_like_celsius, dtype: float64

TOP 20 FEELS LIKE VALUES
feels_like_celsius
21.0    1722
24.6    1695
20.0    1426
24.7    1405
19.0    1391
18.0    1358
24.8    1338
17.0    1196
25.0    1192
25.2    1149
25.1    1141
24.9    1139
24.5    1131
16.0    1125
25.3     976
25.4     972
25.5     959
25.8     887
26.0     881
25.6     876
Name: count, dtype: int64

MAX TEMPERATURE
42.8

MAX FEELS LIKE
50.5

MIN FEELS LIKE
-36.5


INVESTIGATE HEATWAVE INDEX

In [2]:
print("\nHEATWAVE INDEX SUMMARY")
print(df["heatwave_index"].describe())

print("\nCORRELATION WITH FEELS LIKE")
print(
    df[
        ["heatwave_index", "feels_like_celsius", "temperature_celsius", "humidity"]
    ].corr()
)


HEATWAVE INDEX SUMMARY
count    119398.000000
mean         22.709389
std          22.390412
min         -30.700000
25%          16.600000
50%          21.200000
75%          24.900000
max         316.800000
Name: heatwave_index, dtype: float64

CORRELATION WITH FEELS LIKE
                     heatwave_index  feels_like_celsius  temperature_celsius  humidity
heatwave_index             1.000000            0.445117             0.431770  0.039522
feels_like_celsius         0.445117            1.000000             0.983053  0.134282
temperature_celsius        0.431770            0.983053             1.000000  0.031131
humidity                   0.039522            0.134282             0.031131  1.000000


FEELS LIKE DISTRIBUTION

In [3]:
print(df["feels_like_celsius"].quantile([0.50, 0.75, 0.90, 0.95, 0.99]))

print("\nTOP VALUES")
print(df["feels_like_celsius"].sort_values(ascending=False).head(20))

0.50    21.2
0.75    26.2
0.90    29.2
0.95    31.0
0.99    35.3
Name: feels_like_celsius, dtype: float64

TOP VALUES
9037      50.5
118717    47.2
775       47.0
8868      44.7
114335    44.4
119344    44.3
119005    44.1
118168    44.1
114201    43.9
118904    43.8
119134    43.5
119315    43.4
119321    43.4
119347    43.4
8933      43.4
118919    43.4
8954      43.4
9273      43.2
115293    43.2
119318    43.2
Name: feels_like_celsius, dtype: float64


CREATE HEATWAVE TARGET

In [4]:
bins = [float("-inf"), 26.2, 31.0, float("inf")]

labels = ["Safe", "Warning", "Critical"]

df["heatwave_risk"] = pd.cut(df["feels_like_celsius"], bins=bins, labels=labels)

print(df["heatwave_risk"].value_counts())

print("\nPercentage Distribution")

print(round(df["heatwave_risk"].value_counts(normalize=True) * 100, 2))

heatwave_risk
Safe        89967
Warning     23476
Critical     5955
Name: count, dtype: int64

Percentage Distribution
heatwave_risk
Safe        75.35
Warning     19.66
Critical     4.99
Name: proportion, dtype: float64


C:\Users\ANAMIKA\AppData\Local\Temp\ipykernel_4056\712794520.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["heatwave_risk"] = pd.cut(df["feels_like_celsius"], bins=bins, labels=labels)


REMOVE LEAKAGE FEATURES

In [5]:
condition_cols = [col for col in df.columns if col.startswith("condition_text_")]

remove_cols = [
    "feels_like_celsius",
    "climate_cluster",
    "climate_profile",
    "anomaly_flag",
    "anomaly_status",
    "anomaly_score",
] + condition_cols

X = df.drop(columns=remove_cols + ["heatwave_risk"])

y = df["heatwave_risk"]

print("Condition Columns Removed:", len(condition_cols))
print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nTarget Distribution")
print(y.value_counts())

Condition Columns Removed: 49
Feature Matrix Shape: (119398, 94)
Target Shape: (119398,)

Target Distribution
heatwave_risk
Safe        89967
Warning     23476
Critical     5955
Name: count, dtype: int64


TRAIN TEST SPLIT

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("X Train:", X_train.shape)
print("X Test :", X_test.shape)

print("\nTrain Distribution")
print(round(y_train.value_counts(normalize=True) * 100, 2))

print("\nTest Distribution")
print(round(y_test.value_counts(normalize=True) * 100, 2))

X Train: (95518, 94)
X Test : (23880, 94)

Train Distribution
heatwave_risk
Safe        75.35
Warning     19.66
Critical     4.99
Name: proportion, dtype: float64

Test Distribution
heatwave_risk
Safe        75.35
Warning     19.66
Critical     4.99
Name: proportion, dtype: float64


LABEL ENCODING

In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

print("Classes:")
print(le.classes_)

print("\nClass Mapping")

for i, cls in enumerate(le.classes_):
    print(i, "->", cls)

Classes:
['Critical' 'Safe' 'Warning']

Class Mapping
0 -> Critical
1 -> Safe
2 -> Warning


BASELINE LOGISTIC REGRESSION

In [8]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42)

lr_model.fit(X_train, y_train_encoded)

print("Logistic Regression Trained Successfully")

Logistic Regression Trained Successfully


c:\Projects\ClimateGuardAI\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LOGISTIC REGRESSION EVALUATION

In [9]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = lr_model.predict(X_test)

print("Predictions Generated")

print("\nAccuracy:")
print(round(accuracy_score(y_test_encoded, y_pred), 4))

print("\nClassification Report")

print(classification_report(y_test_encoded, y_pred, target_names=le.classes_))

print("\nConfusion Matrix")

print(confusion_matrix(y_test_encoded, y_pred))

Predictions Generated

Accuracy:
0.9534

Classification Report
              precision    recall  f1-score   support

    Critical       0.90      1.00      0.95      1191
        Safe       1.00      0.95      0.97     17994
     Warning       0.83      0.96      0.89      4695

    accuracy                           0.95     23880
   macro avg       0.91      0.97      0.94     23880
weighted avg       0.96      0.95      0.95     23880


Confusion Matrix
[[ 1191     0     0]
 [    0 17052   942]
 [  137    33  4525]]


LEAKAGE AUDIT

In [10]:
corr_cols = [
    "temperature_celsius",
    "temperature_gap",
    "heatwave_index",
    "humidity",
    "uv_index",
]

print(
    df[corr_cols + ["feels_like_celsius"]]
    .corr()["feels_like_celsius"]
    .sort_values(ascending=False)
)

feels_like_celsius     1.000000
temperature_celsius    0.983053
temperature_gap        0.655719
heatwave_index         0.445117
uv_index               0.168632
humidity               0.134282
Name: feels_like_celsius, dtype: float64


FINAL FEATURE MATRIX

In [ ]:
condition_cols = [col for col in df.columns if col.startswith("condition_text_")]

remove_cols = [
    "feels_like_celsius",
    "temperature_gap",
    "climate_cluster",
    "climate_profile",
    "anomaly_flag",
    "anomaly_status",
    "anomaly_score",
] + condition_cols

X = df.drop(columns=remove_cols + ["heatwave_risk"])

y = df["heatwave_risk"]

print("Final Feature Matrix")
print(X.shape)

print("\nTarget")
print(y.shape)

Final Feature Matrix
(119398, 93)

Target
(119398,)


TRAIN TEST SPLIT

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("X Train:", X_train.shape)
print("X Test :", X_test.shape)

print("\nTrain Distribution")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTest Distribution")
print((y_test.value_counts(normalize=True) * 100).round(2))

X Train: (95518, 93)
X Test : (23880, 93)

Train Distribution
heatwave_risk
Safe        75.35
Warning     19.66
Critical     4.99
Name: proportion, dtype: float64

Test Distribution
heatwave_risk
Safe        75.35
Warning     19.66
Critical     4.99
Name: proportion, dtype: float64


RANDOM FOREST BASELINE

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced"
)

rf_model.fit(X_train, y_train)

print("Random Forest Trained Successfully")

Random Forest Trained Successfully


RANDOM FOREST EVALUATION

In [14]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred_rf = rf_model.predict(X_test)

print("Predictions Generated Successfully")

print("\nAccuracy:")
print(round(accuracy_score(y_test, y_pred_rf), 4))

print("\nClassification Report")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_rf))

train_pred_rf = rf_model.predict(X_train)

print("\nTrain Accuracy:", round(accuracy_score(y_train, train_pred_rf), 4))

Predictions Generated Successfully

Accuracy:
0.9809

Classification Report
              precision    recall  f1-score   support

    Critical       0.93      0.94      0.94      1191
        Safe       1.00      0.98      0.99     17994
     Warning       0.93      0.98      0.95      4695

    accuracy                           0.98     23880
   macro avg       0.95      0.97      0.96     23880
weighted avg       0.98      0.98      0.98     23880


Confusion Matrix
[[ 1119     0    72]
 [    0 17723   271]
 [   83    31  4581]]

Train Accuracy: 0.9983


FEATURE IMPORTANCE

In [15]:
import pandas as pd

feature_importance = pd.DataFrame(
    {"Feature": X.columns, "Importance": rf_model.feature_importances_}
)

feature_importance = feature_importance.sort_values(by="Importance", ascending=False)

print(feature_importance.head(25))

                         Feature  Importance
2            temperature_celsius    0.268782
31                heatwave_index    0.226669
26            day_length_minutes    0.056495
5                    pressure_mb    0.048812
7                       humidity    0.034701
0                       latitude    0.029648
30    humidity_cloud_interaction    0.021813
22                         month    0.019706
24                          hour    0.018522
29     wind_humidity_interaction    0.017503
8                          cloud    0.017142
1                      longitude    0.015505
92                      season_3    0.014758
9                  visibility_km    0.012470
16             air_quality_PM2.5    0.011230
27                 pm_difference    0.010425
17              air_quality_PM10    0.009710
4                    wind_degree    0.009168
10                      uv_index    0.009082
14  air_quality_Nitrogen_dioxide    0.008881
28           pollution_intensity    0.008687
15   air_q

Train XGBoost Heatwave Model.

In [17]:
from sklearn.preprocessing import LabelEncoder

heatwave_encoder = LabelEncoder()

y_encoded = heatwave_encoder.fit_transform(y)

print("Class Mapping")

for i, cls in enumerate(heatwave_encoder.classes_):
    print(i, "->", cls)

Class Mapping
0 -> Critical
1 -> Safe
2 -> Warning


Train-Test Split Again

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_enc, y_test_enc = train_test_split(
    X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)

print(X_train.shape)
print(X_test.shape)

(95518, 93)
(23880, 93)


Train XGBoost

In [19]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
)

xgb_model.fit(X_train, y_train_enc)

print("XGBoost Trained Successfully")

XGBoost Trained Successfully


XGBOOST EVALUATION

In [20]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred_xgb = xgb_model.predict(X_test)

print("Predictions Generated")

print("\nAccuracy:")
print(round(accuracy_score(y_test_enc, y_pred_xgb), 4))

print("\nClassification Report")
print(
    classification_report(
        y_test_enc, y_pred_xgb, target_names=heatwave_encoder.classes_
    )
)

print("\nConfusion Matrix")
print(confusion_matrix(y_test_enc, y_pred_xgb))

train_pred_xgb = xgb_model.predict(X_train)

print("\nTrain Accuracy:", round(accuracy_score(y_train_enc, train_pred_xgb), 4))

Predictions Generated

Accuracy:
0.9906

Classification Report
              precision    recall  f1-score   support

    Critical       0.95      0.95      0.95      1191
        Safe       1.00      1.00      1.00     17994
     Warning       0.97      0.98      0.98      4695

    accuracy                           0.99     23880
   macro avg       0.98      0.98      0.98     23880
weighted avg       0.99      0.99      0.99     23880


Confusion Matrix
[[ 1134     0    57]
 [    0 17929    65]
 [   54    49  4592]]

Train Accuracy: 0.997


Save Final Model

In [21]:
import joblib

joblib.dump(xgb_model, "../models/xgboost_heatwave_model.pkl")

print("Heatwave Model Saved Successfully")

Heatwave Model Saved Successfully


Save Feature Names

In [22]:
feature_names = X.columns.tolist()

joblib.dump(feature_names, "../models/heatwave_feature_names.pkl")

print("Feature Names Saved Successfully")
print(len(feature_names))

Feature Names Saved Successfully
93


Save Label Encoder

In [23]:
joblib.dump(heatwave_encoder, "../models/heatwave_label_encoder.pkl")

print("Heatwave Label Encoder Saved")

Heatwave Label Encoder Saved


Save Prediction Dataset

In [27]:
df.to_csv("../datasets/processed/climate_heatwave_predictions.csv", index=False)

print("Heatwave Dataset Saved Successfully")

Heatwave Dataset Saved Successfully
